# Project 03: Gibbs Sampling (Random Algorithm)

In [ ]:
import random
import numpy as np
import bamnostic as bs
import seqlogo as sl

# import function for building sequence motif & identifying seqs matching to motif
from data_readers import *
from seq_ops import get_seq, reverse_complement
from motif_ops import *

---## Implement Gibbs Sampler

Gibbs sampling is a MCMC approach to identify enrichments. Here we will implement a method to identify motifs from a set of regions.Important considerations:- We will need to score each sequence with a PWM using the `score_kmer()` or `score_sequence()` functions    - You will need to investigate into the help documenation and libraries to identify how best to use these functions.- These sites are often not strand-specific and so both scores on the negative as well as positive strand should be considered- To select a random sequence, use `random.randint()` or `numpy.random.randint()`- To select a new position $m$ (as defined below) use `random.choices()` or `numpy.random.choice()`Assumptions:- We know $k$ as the length of expected motif- Each sequence contains the motif

```GibbsMotifFinder(DNA, k-length)    random pick of k-length sequences from each line of DNA as Motifs    for j ← 1 to 10000 or Motifs stops changing        i ← Random(N) where N is number of DNA entries        PWM ← PWM constructed from all Motifs except for Motifi        Motifi ← select position m from PWM-scored k-mers in DNAi in probabilistic fashion from score distribution    return PFM```Probability of chosing position $m = \frac{A_{m}}{\sum_{l}A_{l}}$ for positions $l$ in DNAi

**Note:** I have also added a function to `motif_ops.py` that will calculate the information content of your motifs. This is useful to observe the progression of your Gibbs sampler as well as a measure of convergence. You can use this function as `IC = pfm_ic(pfm)`. You should expect a slow increase of IC until it plateaus such as in the plot below from your lecture slides:<center><img src='figures/Gibbs_Sampling.png'/ width=600px></center>

---# Project Start

In [ ]:
def init_motifs(seqs, k, rng):
    """
    Pick a random k-mer from each sequence to start the chain.
    Pick a random strand label, forward or reverse, a way to add noise to the data.

    Args:
        seqs: list of DNA strings, can be different lengths
        k: motif width
        rng: numpy random generator

    Returns:
        motifs: list of k-mer strings (reverse complemented if strand is R)
        strands: list of 'F' or 'R' labels, one per sequence
        positions: list of start positions in the original sequences
    """
    motifs = []
    strands = []
    positions = []

    for seq in seqs:
        # random start that still fits a k-mer inside the sequence
        pos = rng.integers(0, len(seq) - k + 1)
        kmer = seq[pos : pos + k]

        # Random assignment of strand label
        if rng.integers(0, 2) == 1:
            kmer = reverse_complement(kmer)
            strand_label = 'R'
        else:
            strand_label = 'F'

        motifs.append(kmer)
        strands.append(strand_label)
        positions.append(pos)

    return motifs, strands, positions

In [ ]:
def init_pwm(other_motifs, k):
    """
    Build a PWM from the motifs we are keeping (everything except the holdout).
    Just chains the two template functions together.

    Args:
        other_motifs: k-mer strings from every sequence except the holdout
        k: motif width

    Returns:
        pwm: 4 x k numpy array in log2 odds
    """
    pfm = build_pfm(other_motifs, k)
    pwm = build_pwm(pfm)
    return pwm

In [ ]:
def slide_and_score(holdout_seq, pwm, k):
    """
    Walk a sliding window along the holdout sequence and score every k-mer
    against the PWM, both forward and reverse complement. Keep both scores
    separately, do NOT take the max. Max cuts the data in half and amplifies
    the AGGAGG filter bias.

    Scores and strand labels come out interleaved: F0, R0, F1, R1, ...
    so each pair of neighbours in the output corresponds to one position.

    Args:
        holdout_seq: the full-length sequence we are scoring
        pwm: 4 x k log2 odds matrix
        k: motif width

    Returns:
        scores: list of floats, length 2 * (L - k + 1)
        strand_labels: list of 'F' or 'R', same length as scores
    """
    scores = []
    strand_labels = []

    # slide a k-wide window across every valid start position in the holdout
    for p in range(0, len(holdout_seq) - k + 1):
        fwd_kmer = holdout_seq[p : p + k]
        rev_kmer = reverse_complement(fwd_kmer)

        # score both strands at this position
        fwd_score = score_kmer(fwd_kmer, pwm)
        rev_score = score_kmer(rev_kmer, pwm)

        # append in interleaved order so pos p lands at indices 2p and 2p+1
        scores.append(fwd_score)
        strand_labels.append('F')
        scores.append(rev_score)
        strand_labels.append('R')

    return scores, strand_labels

In [ ]:
def sample_new_motif(scores, strand_labels, holdout_seq, k, rng):
    """
    Turn the scores into a probability distribution and pick one position.
    The "probabilistic sampling" step, so the chain can still explore even
    when a candidate looks obviously best.

    Args:
        scores: interleaved F/R scores from slide_and_score
        strand_labels: matching 'F'/'R' labels
        holdout_seq: the full-length sequence we are sampling from
        k: motif width
        rng: numpy random generator

    Returns:
        picked_motif: the chosen k-mer (reverse complemented if strand is R)
        picked_strand: 'F' or 'R'
        picked_position: start position in the holdout sequence
    """
    # exponentiate with base 2 since the PWM is already in log2 space
    odds = 2 ** np.array(scores)
    probs = odds / odds.sum()

    # weighted draw: higher-scoring positions get picked more often
    # low-scoring ones still have a chance, which is what keeps the chain exploring
    picked_idx = rng.choice(len(probs), p=probs)

    # recover the position and strand from the interleaved index
    picked_position = picked_idx // 2
    picked_strand = strand_labels[picked_idx]

    picked_motif = holdout_seq[picked_position : picked_position + k]
    if picked_strand == 'R':
        picked_motif = reverse_complement(picked_motif)

    return picked_motif, picked_strand, picked_position

In [ ]:
def check_convergence(current_ic, previous_ic, streak, required_streak, tol):
    """
    Streak-based convergence check: count how many iterations in a row
    the IC has barely moved. If we hit the required streak, call it converged.
    "until IC doesn't change" criterion

    Args:
        current_ic: IC after this iteration
        previous_ic: IC from the last iteration, or None on the first call
        streak: current streak count going in
        required_streak: how many stable iterations we want before stopping
        tol: how small the IC change has to be to count as stable (bits)

    Returns:
        new_streak: updated streak count
        converged: True if we have hit the required streak
    """
    # first iteration has no previous value to compare against
    if previous_ic is None:
        return 0, False

    # if IC moved a lot, reset the streak; otherwise bump it up
    if abs(current_ic - previous_ic) < tol:
        new_streak = streak + 1
    else:
        new_streak = 0

    converged = new_streak >= required_streak
    return new_streak, converged

In [ ]:
def GibbsMotifFinder(seqs, k, seed=None, max_iter=10000,
                     required_streak=50, tol=0.001, min_iter=2000):
    """
    Find a shared motif of width k across a list of sequences using
    Gibbs sampling. The algorithm starts with random motifs and iteratively
    refines them one at a time until the PFM stabilizes.

    Args:
        seqs: list of DNA strings
        k: motif width
        seed: random seed, use None for truly random
        max_iter: hard cap on iterations (safety valve)
        required_streak: consecutive stable iterations before declaring converged
        tol: how small an IC change counts as stable (bits)
        min_iter: floor on iterations before convergence can trigger,
                  important for large N where per-iteration change is tiny

    Returns:
        pfm: 4 x k frequency matrix from the final set of motifs
    """
    random.seed(seed)
    rng = np.random.default_rng(seed)

    # bail early if any sequence is too short to hold a k-mer
    too_short = [(i, len(s)) for i, s in enumerate(seqs) if len(s) < k]
    if too_short:
        raise ValueError(f"{len(too_short)} sequence(s) shorter than k={k}: {too_short}")

    # start with random k-mers and random strands, this is just noise
    motifs, strands, positions = init_motifs(seqs, k, rng)

    # convergence tracking state
    streak = 0
    previous_ic = None

    # main Gibbs loop: hold out one sequence, build PWM from the rest,
    # score every k-mer in the holdout, sample a new motif probabilistically
    for iteration in range(max_iter):

        # pick which sequence to update this round
        i = rng.integers(0, len(seqs))

        # build PWM from everyone else (do NOT include the holdout, that would be circular)
        other_motifs = [motifs[j] for j in range(len(motifs)) if j != i]
        pwm = init_pwm(other_motifs, k)

        # score every position in the full-length holdout sequence (seqs[i], not motifs[i])
        scores, strand_labels = slide_and_score(seqs[i], pwm, k)

        # sample a new motif from the score distribution and update our state
        picked_motif, picked_strand, picked_position = sample_new_motif(
            scores, strand_labels, seqs[i], k, rng
        )
        motifs[i] = picked_motif
        strands[i] = picked_strand
        positions[i] = picked_position

        # compute IC of the current full PFM so we can watch convergence
        current_pfm = build_pfm(motifs, k)
        current_ic = pfm_ic(current_pfm)

        # progress print, change the modulo if you want more or less output
        if iteration % 500 == 0:
            print(f"Iter {iteration}: IC = {current_ic:.3f} bits")

        # update the streak and bail if we have hit the required streak
        # not before min_iter, since large N can give false convergence early
        streak, converged = check_convergence(
            current_ic, previous_ic, streak, required_streak, tol
        )
        if converged and iteration >= min_iter:
            print(f"Converged after {iteration + 1} iterations")
            break

        previous_ic = current_ic

    else:
        # for-else runs only if the loop finished without hitting break
        print(f"Hit max_iter={max_iter} without converging")

    # final PFM from whatever motifs we ended up with
    final_pfm = build_pfm(motifs, k)
    return final_pfm

In [ ]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

# read promoters, store in a list of strings
seq_file = "data/GCF_000009045.1_ASM904v1_genomic.fna"
gff_file = "data/GCF_000009045.1_ASM904v1_genomic.gff"

seqs = []

for name, seq in get_fasta(seq_file):  # For each entry in our FASTA file
    for gff_entry in get_gff(gff_file):  # For each entry in our GFF file
        if gff_entry.type == 'CDS':  # If this is a coding sequence
            promoter_seq = get_seq(seq, gff_entry.start, gff_entry.end, gff_entry.strand, 50)  # Extract 50 bp as a promoter

            """
            Because the gibbs sampling assumption is broken in just using promoters,
            and because it takes very long time to randomly progress through so many
            regions, for this example we will pre-filter for sequences that all contain
            part of the shine-dalgarno motif:
            """
            if "AGGAGG" in promoter_seq:
                seqs.append(promoter_seq)

---# Driver ProgramDon't change any of the code here. If you have completed the project by following the coding by contract, the following code should work.

In [ ]:
# Run the gibbs sampler:
promoter_pfm = GibbsMotifFinder(seqs, 10, seed=42)

# Plot the final pfm that is generated:
sl.seqlogo(sl.CompletePm(pfm=promoter_pfm.T))

---# Challenge YourselfNow that you potentially have a working skeleton of the project, try to work with the new NRF1 ChIP-seq data: `data/nrf1_gibbs.fa`.Upon inspection, you will see that it is just a FASTA file. Since it is a FASTA file, you already have an example in the code above on how to access FASTA files. It is up to you to do the correct data ingest with it, though.

---# NRF1 Driver ProgramModify the code below and test your Gibbs Sampler.

In [ ]:
# NRF1 is a ChIP-seq dataset with no pre-filter, so the AGGAGG trick does not apply.
# The sequences are uniformly 75 bp, ACGT only, around 90k of them.
nrf1_file = "data/nrf1_gibbs.fa"

nrf1_peaks = []
for name, seq in get_fasta(nrf1_file):
    nrf1_peaks.append(seq)

# Run the gibbs sampler:
nrf1_pfm = GibbsMotifFinder(nrf1_peaks, 10, seed=42)

# Plot the final pfm that is generated:
sl.seqlogo(sl.CompletePm(pfm=nrf1_pfm.T))